# Model_ENSOClim_CoeffCheck

Caroline Juang, c.juang@columbia.edu

February 2023


**This check deals with the average of the current-season gSST. This check feeds into the model `Model_ENSOclim_AkaikeCoeff`.**

In [1]:
# import
import numpy as np
import pandas as pd
from scipy.stats import pearsonr
from scipy.stats import linregress

# User input

In [2]:
# customize seasons for climate variables

# customize number of rolling periods
ant_years = 1 # antecedent years to include (2 antecedent years + current year)
ant_season = 3 # n+1 of months to include in each period (e.g. input 2 would mean 3 months)

firstyear = 1984 # first year of data
finalyear = 2022 # final year of data (should be same as burned area)
time_length = int(finalyear-firstyear+1) # get length of timeseries

# SST gradient: (125-155 deg lon) - nino3.4 (190-240 deg)
# UPDATE THIS FOR THE CORRECT SST GRADIENT
# Read the climate index from the .txt file
with open('0_climindname.txt', 'r') as f:
    climindnameObs = f.read().strip()
print(climindnameObs +' will be used for the SST gradient')
climindnameDT = 'DeTrend_'+climindnameObs

# importing data strings
directory = 'your_directory'
#data_string = 'data\\'
data_string = 'data//'
#model_string = 'model\\'+climindnameObs+'\\'
model_string = 'model//'+climindnameObs+'//'

patch125-155_nino3-34 will be used for the SST gradient


# Scripts

In [3]:
# average climate variables, within a selected season

def annualDetrend(data):
    """
    This intakes an array of current ecoregion's climate variable of annual averages (data), 
    Output: an array of the yearly data, detrended so the slope of the data is zero.
    Requirements: 
    """

    # detrend the data
    xnum = np.arange(0,len(data))
    reg = linregress(xnum, data)
    m = reg.slope
    b = reg.intercept
    regpredict = m*xnum + b
    # normalize by subtracting the linear regression
    tmpdatanorm = (data - regpredict) + b

    return tmpdatanorm

# Import data
* Ecoregions as manual input
* **climate** from `Data_CreateModelData`
* **SST gradient** from `Data_CreateENSOIndex` &rarr; `Data_CreateModelData`

In [4]:
# create strings of the types of forests, remove #5
province_names = ['1: American Semi-Desert and Desert Province',
                  '2: Arizona-New Mexico Mountains Semi-Desert-Open Woodland-Coniferous Forest-Alpine Meadow Province',
                  '3: Black Hills Coniferous Forest Province',
                  '4: California Coastal Chapparral Forest and Shrub Province',
                  '5: California Coastal Range Open Woodland-Shrub-Coniferous Forest-Meadow Province',
                  '6: California Coastal Steppe-Mixed Forest-Redwood Forest Province',
                  '7: California Dry Steppe Province',
                  '8: Cascade Mixed Forest-Coniferous Forest-Alpine Meadow Province',
                  '9: Chihuahuan Semi-Desert Province',
                  '10: Colorado Plateau Semi-Desert Province',
                  '12: Great Plains-Palouse Dry Steppe Province',
                  '13: Intermountain Semi-Desert Province',
                  '14: Intermountain Semi-Desert and Desert Province',
                  '15: Middle Rocky Mountain Steppe-Coniferous Forest-Alpine Meadow Province',
                  '16: Nevada-Utah Mountains-Semi-Desert-Coniferous Forest-Alpine Meadow Province',
                  '17: Northern Rocky Mountain Forest-Steppe-Coniferous Forest-Alpine Meadow Province',
                  '18: Pacific Lowland Mixed Forest Province',
                  '19: Sierran Steppe-Mixed Forest-Coniferous Forest-Alpine Meadow Province',
                  '20: Southern Rocky Mountain Steppe-Open Woodland-Coniferous Forest-Alpine Meadow Province',
                  '21: Southwest Plateau and Plains Dry Steppe and Shrub Province']
province_num = [item for item in range(len(province_names)+1+1)]
province_num.remove(11) # remove empty ecoregion - no overlap btwn westUS map and ecoregion
dfnames = ['allwestUS', 'ecoprov1', 'ecoprov2', 'ecoprov3', 'ecoprov4', 'ecoprov5', 
           'ecoprov6', 'ecoprov7', 'ecoprov8', 'ecoprov9', 'ecoprov10', 
           'ecoprov12','ecoprov13','ecoprov14','ecoprov15',
           'ecoprov16','ecoprov17','ecoprov18','ecoprov19','ecoprov20','ecoprov21']

In [5]:
# import model data for climate

dfframesall = {}
dfframesfor = {}
dfframesnon = {}

for iecoreg in np.arange(len(province_num)):
    filename = data_string + 'climatefull_ecoprovinces_'
    dfframesall[dfnames[iecoreg]] = pd.read_csv(filename + dfnames[iecoreg]+'_all.txt').set_index('Unnamed: 0')
    dfframesfor[dfnames[iecoreg]] = pd.read_csv(filename + dfnames[iecoreg]+'_for.txt').set_index('Unnamed: 0')
    dfframesnon[dfnames[iecoreg]] = pd.read_csv(filename + dfnames[iecoreg]+'_non.txt').set_index('Unnamed: 0')

In [6]:
# read in the SST gradient data

# import
climindseasons83Obs = pd.read_csv(data_string + 'sstgrad_seasonfull_' +climindnameObs+'_82_y.txt').set_index('Unnamed: 0')
climindseasons83DT = pd.read_csv(data_string + 'sstgrad_seasonfull_' +climindnameDT+'_82_y.txt').set_index('Unnamed: 0')

# create prior-season variable for y0 mo 1-3
tmppriorObs = climindseasons83Obs['y0 mo 10-12'].loc[firstyear-2:finalyear-1].values
tmppriorDT = climindseasons83DT['y0 mo 10-12'].loc[firstyear-2:finalyear-1].values
# cut to same timeframe as climate data
climindseasons83Obs = climindseasons83Obs.loc[firstyear-1:]
climindseasons83DT = climindseasons83DT.loc[firstyear-1:]
# insert prior-year var
climindseasons83Obs.insert(0, 'y-1 mo 10-12', tmppriorObs)
climindseasons83DT.insert(0, 'y-1 mo 10-12', tmppriorDT)

print(climindnameObs+' imported')
print(climindnameDT+' imported')

patch125-155_nino3-34 imported
DeTrend_patch125-155_nino3-34 imported


# All area

In [7]:
# iterate through each ecoregion, make a list of SST gradients to EXCLUDE

modeloutputfile = model_string + 'coeffExclude_sst_all_ecoprovinces.txt'
f = open(modeloutputfile, 'w') # print to this file
labels = dfframesall['allwestUS'].columns.values # each climate variable in the columns

for iregion, ecoregname in enumerate(dfnames):
    print('++++++++ ALL model for ' + dfnames[iregion]+ '\t r-value obs \t r-value DT ++++++')
    f.write('+++'+dfnames[iregion]+'\n')
    for i, label in enumerate(labels): # iterate through climate variables
        print('PREDICTING '+label)
        f.write('PREDICTING '+label)
        f.write('\n')
        # variables to compare
        # each climate variable
        tmpclim = np.asarray(dfframesall[dfnames[iregion]][label])
        tmpclimDT = annualDetrend(tmpclim)
        # associated concurrent-season climate index (ENSO or other)
        strseason = labels[i].split(' ',1)[1] # get season label
        matchseason = climindseasons83Obs.columns.isin([strseason]) # True/False array
        tmpclimindObs = np.asarray(climindseasons83Obs.loc[:,matchseason]).flatten()
        tmpclimindDT = np.asarray(climindseasons83DT.loc[:,matchseason]).flatten()
        
        # correlation of clim vs. SST gradient
        tmppearsonObs = pearsonr(tmpclimindObs, tmpclim)
        tmprObs = tmppearsonObs.statistic
        tmppObs = tmppearsonObs.pvalue
        tmppearsonDT = pearsonr(tmpclimindDT, tmpclimDT)
        tmprDT = tmppearsonDT.statistic
        tmppDT = tmppearsonDT.pvalue
        # check if they are the same sign
        tmpsign = ((tmprObs > 0) == (tmprDT > 0))
        tmppvalue = (tmppDT < 0.1) # check significance
        print(climindseasons83Obs.columns[matchseason][0] + 
            '\t observed SST: r={:.5} \tSST no trend: r={:.5}'.format(tmprObs, tmprDT))
        print('\t\t observed SST: p={:.5} \tSST no trend: p={:.5}'.format(tmppObs, tmppDT))
        
        if tmpsign==False:
            print('\tSIGN CHANGE ' + climindseasons83Obs.columns[matchseason][0] + '\t'+
                  'r={:.5f} \t r={:.5f}'.format(tmprObs, tmprDT))
        if tmppvalue==False:
            print('\tDT INSIGNIFICANT ' + climindseasons83Obs.columns[matchseason][0] + '\t'+
                  'p={:.5f} \t p={:.5f}'.format(tmppObs, tmppDT))
        if (tmpsign==False) or (tmppvalue==False):
            f.write(str(climindseasons83Obs.columns[matchseason][0]) + '\n')

        
        # grab previous season            
        iprevseason = np.where(matchseason)
        iprevseason = int(iprevseason[0])-1
        tmpclimindObs = np.asarray(climindseasons83Obs.iloc[:,iprevseason]).flatten()
        tmpclimindDT = np.asarray(climindseasons83DT.iloc[:,iprevseason]).flatten()
        # correlation of clim vs. SST gradient
        tmppearsonObs = pearsonr(tmpclimindObs, tmpclim)
        tmprObs = tmppearsonObs.statistic
        tmppObs = tmppearsonObs.pvalue
        tmppearsonDT = pearsonr(tmpclimindDT, tmpclimDT)
        tmprDT = tmppearsonDT.statistic
        tmppDT = tmppearsonDT.pvalue
        
        # check if previous season changed correlation sign
        tmpsign = ((tmprObs > 0) == (tmprDT > 0))
        tmppvalue = (tmppDT < 0.1) # check significance
        print(climindseasons83Obs.columns[iprevseason] + 
              '\t observed SST: r={:.5} \tSST no trend: r={:.5}'.format(tmprObs, tmprDT))
        print('\t\t observed SST: p={:.5} \tSST no trend: p={:.5}'.format(tmppObs, tmppDT))
        
        if tmpsign==False:
            print('\tSIGN CHANGE ' + climindseasons83Obs.columns[iprevseason] + '\t'+
                  'r={:.5f} \t r={:.5f}'.format(tmprObs, tmprDT))
            print('\t\t\t observed SST: p={:.5} \tSST no trend: p={:.5}'.format(tmppObs, tmppDT))
        if tmppvalue==False:
            print('\tDT INSIGNIFICANT ' + climindseasons83Obs.columns[iprevseason] + '\t'+
                  'p={:.5f} \t p={:.5f}'.format(tmppObs, tmppDT))
        if (tmpsign==False) or (tmppvalue==False):
            f.write(str(climindseasons83Obs.columns[iprevseason])+'\n')
f.close()

++++++++ ALL model for allwestUS	 r-value obs 	 r-value DT ++++++
PREDICTING rh y0 mo 1-3
y0 mo 1-3	 observed SST: r=-0.58462 	SST no trend: r=-0.53402
		 observed SST: p=7.4539e-05 	SST no trend: p=0.00038668
y-1 mo 10-12	 observed SST: r=-0.45937 	SST no trend: r=-0.4138
		 observed SST: p=0.0028653 	SST no trend: p=0.0079487
PREDICTING rh y0 mo 4-6
y0 mo 4-6	 observed SST: r=-0.46425 	SST no trend: r=-0.4103
		 observed SST: p=0.0025474 	SST no trend: p=0.0085485
y0 mo 1-3	 observed SST: r=-0.56806 	SST no trend: r=-0.51395
		 observed SST: p=0.00013162 	SST no trend: p=0.00069334
PREDICTING rh y0 mo 7-9
y0 mo 7-9	 observed SST: r=-0.20707 	SST no trend: r=-0.10719
		 observed SST: p=0.19983 	SST no trend: p=0.51031
	DT INSIGNIFICANT y0 mo 7-9	p=0.19983 	 p=0.51031
y0 mo 4-6	 observed SST: r=-0.35192 	SST no trend: r=-0.26566
		 observed SST: p=0.02595 	SST no trend: p=0.097549
PREDICTING rh y0 mo 10-12
y0 mo 10-12	 observed SST: r=-0.28107 	SST no trend: r=-0.2379
		 observed SST: 

y0 mo 1-3	 observed SST: r=0.42959 	SST no trend: r=0.38078
		 observed SST: p=0.0056688 	SST no trend: p=0.015349
y-1 mo 10-12	 observed SST: r=0.35228 	SST no trend: r=0.31068
		 observed SST: p=0.02579 	SST no trend: p=0.051035
PREDICTING solar y0 mo 4-6
y0 mo 4-6	 observed SST: r=0.1799 	SST no trend: r=0.10846
		 observed SST: p=0.26666 	SST no trend: p=0.50529
	DT INSIGNIFICANT y0 mo 4-6	p=0.26666 	 p=0.50529
y0 mo 1-3	 observed SST: r=0.16944 	SST no trend: r=0.0822
		 observed SST: p=0.29591 	SST no trend: p=0.61409
	DT INSIGNIFICANT y0 mo 1-3	p=0.29591 	 p=0.61409
PREDICTING solar y0 mo 7-9
y0 mo 7-9	 observed SST: r=0.086845 	SST no trend: r=0.098832
		 observed SST: p=0.59414 	SST no trend: p=0.54403
	DT INSIGNIFICANT y0 mo 7-9	p=0.59414 	 p=0.54403
y0 mo 4-6	 observed SST: r=-0.068522 	SST no trend: r=-0.060211
		 observed SST: p=0.6744 	SST no trend: p=0.71208
	DT INSIGNIFICANT y0 mo 4-6	p=0.67440 	 p=0.71208
PREDICTING solar y0 mo 10-12
y0 mo 10-12	 observed SST: r=0.3237

y0 mo 10-12	 observed SST: r=0.0474 	SST no trend: r=-0.047814
		 observed SST: p=0.77148 	SST no trend: p=0.76954
	SIGN CHANGE y0 mo 10-12	r=0.04740 	 r=-0.04781
	DT INSIGNIFICANT y0 mo 10-12	p=0.77148 	 p=0.76954
y0 mo 7-9	 observed SST: r=0.028631 	SST no trend: r=-0.088646
		 observed SST: p=0.86079 	SST no trend: p=0.58648
	SIGN CHANGE y0 mo 7-9	r=0.02863 	 r=-0.08865
			 observed SST: p=0.86079 	SST no trend: p=0.58648
	DT INSIGNIFICANT y0 mo 7-9	p=0.86079 	 p=0.58648
PREDICTING vpd y0 mo 1-3
y0 mo 1-3	 observed SST: r=0.30163 	SST no trend: r=0.2095
		 observed SST: p=0.058554 	SST no trend: p=0.19448
	DT INSIGNIFICANT y0 mo 1-3	p=0.05855 	 p=0.19448
y-1 mo 10-12	 observed SST: r=0.18714 	SST no trend: r=0.10928
		 observed SST: p=0.24756 	SST no trend: p=0.50205
	DT INSIGNIFICANT y-1 mo 10-12	p=0.24756 	 p=0.50205
PREDICTING vpd y0 mo 4-6
y0 mo 4-6	 observed SST: r=0.18795 	SST no trend: r=0.14437
		 observed SST: p=0.24549 	SST no trend: p=0.37412
	DT INSIGNIFICANT y0 mo 4-6	p

/var/folders/bh/cplk6_2163d7cqx4vgqpkzz40000gn/T/ipykernel_58417/2049283274.py:50: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  iprevseason = int(iprevseason[0])-1


y0 mo 4-6	 observed SST: r=0.16343 	SST no trend: r=0.095179
		 observed SST: p=0.31363 	SST no trend: p=0.55908
	DT INSIGNIFICANT y0 mo 4-6	p=0.31363 	 p=0.55908
y0 mo 1-3	 observed SST: r=0.41363 	SST no trend: r=0.35607
		 observed SST: p=0.0079768 	SST no trend: p=0.024129
PREDICTING tmean y0 mo 7-9
y0 mo 7-9	 observed SST: r=0.20505 	SST no trend: r=0.059889
		 observed SST: p=0.20435 	SST no trend: p=0.71355
	DT INSIGNIFICANT y0 mo 7-9	p=0.20435 	 p=0.71355
y0 mo 4-6	 observed SST: r=0.25501 	SST no trend: r=0.10677
		 observed SST: p=0.11228 	SST no trend: p=0.51199
	DT INSIGNIFICANT y0 mo 4-6	p=0.11228 	 p=0.51199
PREDICTING tmean y0 mo 10-12
y0 mo 10-12	 observed SST: r=0.22955 	SST no trend: r=0.16489
		 observed SST: p=0.15419 	SST no trend: p=0.30926
	DT INSIGNIFICANT y0 mo 10-12	p=0.15419 	 p=0.30926
y0 mo 7-9	 observed SST: r=0.21077 	SST no trend: r=0.12847
		 observed SST: p=0.19173 	SST no trend: p=0.4295
	DT INSIGNIFICANT y0 mo 7-9	p=0.19173 	 p=0.42950
PREDICTING vpd

...

y0 mo 10-12	 observed SST: r=0.14503 	SST no trend: r=0.1199
		 observed SST: p=0.37191 	SST no trend: p=0.46115
	DT INSIGNIFICANT y0 mo 10-12	p=0.37191 	 p=0.46115
y0 mo 7-9	 observed SST: r=0.098719 	SST no trend: r=0.067676
		 observed SST: p=0.54449 	SST no trend: p=0.6782
	DT INSIGNIFICANT y0 mo 7-9	p=0.54449 	 p=0.67820
PREDICTING wetdays y0 mo 1-3
y0 mo 1-3	 observed SST: r=-0.17036 	SST no trend: r=-0.1358
		 observed SST: p=0.29325 	SST no trend: p=0.40342
	DT INSIGNIFICANT y0 mo 1-3	p=0.29325 	 p=0.40342
y-1 mo 10-12	 observed SST: r=-0.071337 	SST no trend: r=-0.040731
		 observed SST: p=0.6618 	SST no trend: p=0.80294
	DT INSIGNIFICANT y-1 mo 10-12	p=0.66180 	 p=0.80294
PREDICTING wetdays y0 mo 4-6
y0 mo 4-6	 observed SST: r=-0.37135 	SST no trend: r=-0.35704
		 observed SST: p=0.018313 	SST no trend: p=0.023722
y0 mo 1-3	 observed SST: r=-0.37695 	SST no trend: r=-0.36288
		 observed SST: p=0.016502 	SST no trend: p=0.021371
PREDICTING wetdays y0 mo 7-9
y0 mo 7-9	 observed

y0 mo 4-6	 observed SST: r=0.27003 	SST no trend: r=0.21895
		 observed SST: p=0.091959 	SST no trend: p=0.17465
	DT INSIGNIFICANT y0 mo 4-6	p=0.09196 	 p=0.17465
y0 mo 1-3	 observed SST: r=0.36722 	SST no trend: r=0.31473
		 observed SST: p=0.019757 	SST no trend: p=0.047926
PREDICTING tmax y0 mo 7-9
y0 mo 7-9	 observed SST: r=0.34885 	SST no trend: r=0.26618
		 observed SST: p=0.02737 	SST no trend: p=0.096872
y0 mo 4-6	 observed SST: r=0.40909 	SST no trend: r=0.33089
		 observed SST: p=0.0087656 	SST no trend: p=0.037019
PREDICTING tmax y0 mo 10-12
y0 mo 10-12	 observed SST: r=0.35469 	SST no trend: r=0.31234
		 observed SST: p=0.024724 	SST no trend: p=0.049745
y0 mo 7-9	 observed SST: r=0.3015 	SST no trend: r=0.24507
		 observed SST: p=0.058668 	SST no trend: p=0.12746
	DT INSIGNIFICANT y0 mo 7-9	p=0.05867 	 p=0.12746
PREDICTING tmin y0 mo 1-3
y0 mo 1-3	 observed SST: r=-0.21603 	SST no trend: r=-0.29684
		 observed SST: p=0.18061 	SST no trend: p=0.062883
y-1 mo 10-12	 observed

y0 mo 4-6	 observed SST: r=-0.31344 	SST no trend: r=-0.26417
		 observed SST: p=0.048898 	SST no trend: p=0.099517
y0 mo 1-3	 observed SST: r=-0.44411 	SST no trend: r=-0.3969
		 observed SST: p=0.0040951 	SST no trend: p=0.011222
PREDICTING wetdays y0 mo 7-9
y0 mo 7-9	 observed SST: r=-0.08767 	SST no trend: r=-0.064057
		 observed SST: p=0.59062 	SST no trend: p=0.69455
	DT INSIGNIFICANT y0 mo 7-9	p=0.59062 	 p=0.69455
y0 mo 4-6	 observed SST: r=-0.018167 	SST no trend: r=0.010676
		 observed SST: p=0.91141 	SST no trend: p=0.94787
	SIGN CHANGE y0 mo 4-6	r=-0.01817 	 r=0.01068
			 observed SST: p=0.91141 	SST no trend: p=0.94787
	DT INSIGNIFICANT y0 mo 4-6	p=0.91141 	 p=0.94787
PREDICTING wetdays y0 mo 10-12
y0 mo 10-12	 observed SST: r=-0.13821 	SST no trend: r=-0.10774
		 observed SST: p=0.39504 	SST no trend: p=0.50816
	DT INSIGNIFICANT y0 mo 10-12	p=0.39504 	 p=0.50816
y0 mo 7-9	 observed SST: r=-0.19921 	SST no trend: r=-0.16588
		 observed SST: p=0.21781 	SST no trend: p=0.306

y0 mo 4-6	 observed SST: r=0.43733 	SST no trend: r=0.40069
		 observed SST: p=0.0047743 	SST no trend: p=0.010402
y0 mo 1-3	 observed SST: r=0.54746 	SST no trend: r=0.51372
		 observed SST: p=0.0002562 	SST no trend: p=0.00069781
PREDICTING wind y0 mo 7-9
y0 mo 7-9	 observed SST: r=0.11343 	SST no trend: r=0.13888
		 observed SST: p=0.48587 	SST no trend: p=0.39274
	DT INSIGNIFICANT y0 mo 7-9	p=0.48587 	 p=0.39274
y0 mo 4-6	 observed SST: r=0.10523 	SST no trend: r=0.13393
		 observed SST: p=0.51815 	SST no trend: p=0.40998
	DT INSIGNIFICANT y0 mo 4-6	p=0.51815 	 p=0.40998
PREDICTING wind y0 mo 10-12
y0 mo 10-12	 observed SST: r=0.12579 	SST no trend: r=0.099936
		 observed SST: p=0.43928 	SST no trend: p=0.53951
	DT INSIGNIFICANT y0 mo 10-12	p=0.43928 	 p=0.53951
y0 mo 7-9	 observed SST: r=0.19301 	SST no trend: r=0.16521
		 observed SST: p=0.23276 	SST no trend: p=0.3083
	DT INSIGNIFICANT y0 mo 7-9	p=0.23276 	 p=0.30830
PREDICTING prec y0 mo 1-3
y0 mo 1-3	 observed SST: r=0.22807 	

y0 mo 7-9	 observed SST: r=0.3716 	SST no trend: r=0.29586
		 observed SST: p=0.018229 	SST no trend: p=0.063794
y0 mo 4-6	 observed SST: r=0.35383 	SST no trend: r=0.25198
		 observed SST: p=0.0251 	SST no trend: p=0.11675
	DT INSIGNIFICANT y0 mo 4-6	p=0.02510 	 p=0.11675
PREDICTING tmax y0 mo 10-12
y0 mo 10-12	 observed SST: r=0.37333 	SST no trend: r=0.32198
		 observed SST: p=0.017655 	SST no trend: p=0.042755
y0 mo 7-9	 observed SST: r=0.32234 	SST no trend: r=0.2439
		 observed SST: p=0.042509 	SST no trend: p=0.12936
	DT INSIGNIFICANT y0 mo 7-9	p=0.04251 	 p=0.12936
PREDICTING tmin y0 mo 1-3
y0 mo 1-3	 observed SST: r=-0.17166 	SST no trend: r=-0.24137
		 observed SST: p=0.28953 	SST no trend: p=0.13351
	DT INSIGNIFICANT y0 mo 1-3	p=0.28953 	 p=0.13351
y-1 mo 10-12	 observed SST: r=-0.21061 	SST no trend: r=-0.26291
		 observed SST: p=0.19208 	SST no trend: p=0.10121
	DT INSIGNIFICANT y-1 mo 10-12	p=0.19208 	 p=0.10121
PREDICTING tmin y0 mo 4-6
y0 mo 4-6	 observed SST: r=0.13825

# Forest

In [8]:
# iterate through each ecoregion, make a list of SST gradients to EXCLUDE

modeloutputfile = model_string + 'coeffExclude_sst_for_ecoprovinces.txt'
f = open(modeloutputfile, 'w') # print to this file
labels = dfframesfor['allwestUS'].columns.values # each climate variable in the columns

for iregion, ecoregname in enumerate(dfnames):
    print('++++++++ FOREST model for ' + dfnames[iregion]+ '\t r-value obs \t r-value DT ++++++')
    f.write('+++'+dfnames[iregion]+'\n')
    for i, label in enumerate(labels): # iterate through climate variables
        print('PREDICTING '+label)
        f.write('PREDICTING '+label)
        f.write('\n')
        # variables to compare
        # each climate variable
        tmpclim = np.asarray(dfframesfor[dfnames[iregion]][label])
        tmpclimDT = annualDetrend(tmpclim)
        # associated concurrent-season climate index (ENSO or other)
        strseason = labels[i].split(' ',1)[1] # get season label
        matchseason = climindseasons83Obs.columns.isin([strseason]) # True/False array
        tmpclimindObs = np.asarray(climindseasons83Obs.loc[:,matchseason]).flatten()
        tmpclimindDT = np.asarray(climindseasons83DT.loc[:,matchseason]).flatten()
        
        # correlation of clim vs. SST gradient
        tmppearsonObs = pearsonr(tmpclimindObs, tmpclim)
        tmprObs = tmppearsonObs.statistic
        tmppObs = tmppearsonObs.pvalue
        tmppearsonDT = pearsonr(tmpclimindDT, tmpclimDT)
        tmprDT = tmppearsonDT.statistic
        tmppDT = tmppearsonDT.pvalue
        # check if they are the same sign
        tmpsign = ((tmprObs > 0) == (tmprDT > 0))
        tmppvalue = (tmppDT < 0.1) # check significance
        print(climindseasons83Obs.columns[matchseason][0] + 
            '\t observed SST: r={:.5} \tSST no trend: r={:.5}'.format(tmprObs, tmprDT))
        print('\t\t observed SST: p={:.5} \tSST no trend: p={:.5}'.format(tmppObs, tmppDT))
        
        if tmpsign==False:
            print('\tSIGN CHANGE ' + climindseasons83Obs.columns[matchseason][0] + '\t'+
                  'r={:.5f} \t r={:.5f}'.format(tmprObs, tmprDT))
        if tmppvalue==False:
            print('\tDT INSIGNIFICANT ' + climindseasons83Obs.columns[matchseason][0] + '\t'+
                  'p={:.5f} \t p={:.5f}'.format(tmppObs, tmppDT))
        if (tmpsign==False) or (tmppvalue==False):
            f.write(str(climindseasons83Obs.columns[matchseason][0]) + '\n')
        
        # grab previous season            
        iprevseason = np.where(matchseason)
        iprevseason = int(iprevseason[0])-1
        tmpclimindObs = np.asarray(climindseasons83Obs.iloc[:,iprevseason]).flatten()
        tmpclimindDT = np.asarray(climindseasons83DT.iloc[:,iprevseason]).flatten()
        # correlation of clim vs. SST gradient
        tmppearsonObs = pearsonr(tmpclimindObs, tmpclim)
        tmprObs = tmppearsonObs.statistic
        tmppObs = tmppearsonObs.pvalue
        tmppearsonDT = pearsonr(tmpclimindDT, tmpclimDT)
        tmprDT = tmppearsonDT.statistic
        tmppDT = tmppearsonDT.pvalue
        
        # check if previous season changed correlation sign
        tmpsign = ((tmprObs > 0) == (tmprDT > 0))
        tmppvalue = (tmppDT < 0.1) # check significance
        print(climindseasons83Obs.columns[iprevseason] + 
              '\t observed SST: r={:.5} \tSST no trend: r={:.5}'.format(tmprObs, tmprDT))
        print('\t\t observed SST: p={:.5} \tSST no trend: p={:.5}'.format(tmppObs, tmppDT))
        if tmpsign==False:
            print('\tSIGN CHANGE ' + climindseasons83Obs.columns[iprevseason] + '\t'+
                  'r={:.5f} \t r={:.5f}'.format(tmprObs, tmprDT))
        if tmppvalue==False:
            print('\tDT INSIGNIFICANT ' + climindseasons83Obs.columns[iprevseason] + '\t'+
                  'p={:.5f} \t p={:.5f}'.format(tmppObs, tmppDT))
        if (tmpsign==False) or (tmppvalue==False):
            f.write(str(climindseasons83Obs.columns[iprevseason])+'\n')
f.close()

++++++++ FOREST model for allwestUS	 r-value obs 	 r-value DT ++++++
PREDICTING rh y0 mo 1-3
y0 mo 1-3	 observed SST: r=-0.56608 	SST no trend: r=-0.50614
		 observed SST: p=0.00014056 	SST no trend: p=0.00086183
y-1 mo 10-12	 observed SST: r=-0.42145 	SST no trend: r=-0.36667
		 observed SST: p=0.0067607 	SST no trend: p=0.019954
PREDICTING rh y0 mo 4-6
y0 mo 4-6	 observed SST: r=-0.39374 	SST no trend: r=-0.32328
		 observed SST: p=0.011947 	SST no trend: p=0.041874
y0 mo 1-3	 observed SST: r=-0.51952 	SST no trend: r=-0.45059
		 observed SST: p=0.0005917 	SST no trend: p=0.0035257
PREDICTING rh y0 mo 7-9
y0 mo 7-9	 observed SST: r=-0.1931 	SST no trend: r=-0.080888
		 observed SST: p=0.23255 	SST no trend: p=0.61977
	DT INSIGNIFICANT y0 mo 7-9	p=0.23255 	 p=0.61977
y0 mo 4-6	 observed SST: r=-0.35939 	SST no trend: r=-0.26915
		 observed SST: p=0.022752 	SST no trend: p=0.093075
PREDICTING rh y0 mo 10-12
y0 mo 10-12	 observed SST: r=-0.20736 	SST no trend: r=-0.15323
		 observed SST

y0 mo 10-12	 observed SST: r=-0.22791 	SST no trend: r=-0.1892
		 observed SST: p=0.15725 	SST no trend: p=0.24231
	DT INSIGNIFICANT y0 mo 10-12	p=0.15725 	 p=0.24231
y0 mo 7-9	 observed SST: r=-0.29378 	SST no trend: r=-0.25152
		 observed SST: p=0.065776 	SST no trend: p=0.11745
	DT INSIGNIFICANT y0 mo 7-9	p=0.06578 	 p=0.11745
++++++++ FOREST model for ecoprov2	 r-value obs 	 r-value DT ++++++
PREDICTING rh y0 mo 1-3
y0 mo 1-3	 observed SST: r=-0.64402 	SST no trend: r=-0.59565
		 observed SST: p=7.3351e-06 	SST no trend: p=5.0178e-05
y-1 mo 10-12	 observed SST: r=-0.55323 	SST no trend: r=-0.51549
		 observed SST: p=0.00021353 	SST no trend: p=0.0006638
PREDICTING rh y0 mo 4-6
y0 mo 4-6	 observed SST: r=-0.48926 	SST no trend: r=-0.42928
		 observed SST: p=0.0013561 	SST no trend: p=0.0057071
y0 mo 1-3	 observed SST: r=-0.59005 	SST no trend: r=-0.52902
		 observed SST: p=6.1451e-05 	SST no trend: p=0.00044883
PREDICTING rh y0 mo 7-9


y0 mo 7-9	 observed SST: r=0.0073509 	SST no trend: r=0.080147
		 observed SST: p=0.96409 	SST no trend: p=0.62299
	DT INSIGNIFICANT y0 mo 7-9	p=0.96409 	 p=0.62299
y0 mo 4-6	 observed SST: r=0.045091 	SST no trend: r=0.12953
		 observed SST: p=0.78233 	SST no trend: p=0.42566
	DT INSIGNIFICANT y0 mo 4-6	p=0.78233 	 p=0.42566
PREDICTING rh y0 mo 10-12
y0 mo 10-12	 observed SST: r=-0.4616 	SST no trend: r=-0.42384
		 observed SST: p=0.0027162 	SST no trend: p=0.0064224
y0 mo 7-9	 observed SST: r=-0.39944 	SST no trend: r=-0.34125
		 observed SST: p=0.010666 	SST no trend: p=0.031163
PREDICTING solar y0 mo 1-3
y0 mo 1-3	 observed SST: r=0.44574 	SST no trend: r=0.39608
		 observed SST: p=0.0039453 	SST no trend: p=0.011406
y-1 mo 10-12	 observed SST: r=0.35765 	SST no trend: r=0.31478
		 observed SST: p=0.023464 	SST no trend: p=0.047894
PREDICTING solar y0 mo 4-6
y0 mo 4-6	 observed SST: r=0.18592 	SST no trend: r=0.11276
		 observed SST: p=0.25072 	SST no trend: p=0.48845
	DT INSIGNIFI

/var/folders/bh/cplk6_2163d7cqx4vgqpkzz40000gn/T/ipykernel_58417/1658375194.py:49: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  iprevseason = int(iprevseason[0])-1


y0 mo 1-3	 observed SST: r=0.38748 	SST no trend: r=0.35662
		 observed SST: p=0.0135 	SST no trend: p=0.023897
y-1 mo 10-12	 observed SST: r=0.27948 	SST no trend: r=0.24954
		 observed SST: p=0.080722 	SST no trend: p=0.12045
	DT INSIGNIFICANT y-1 mo 10-12	p=0.08072 	 p=0.12045
PREDICTING solar y0 mo 4-6
y0 mo 4-6	 observed SST: r=0.19227 	SST no trend: r=0.17582
		 observed SST: p=0.2346 	SST no trend: p=0.27783
	DT INSIGNIFICANT y0 mo 4-6	p=0.23460 	 p=0.27783
y0 mo 1-3	 observed SST: r=0.47801 	SST no trend: r=0.47551
		 observed SST: p=0.0018113 	SST no trend: p=0.0019292
PREDICTING solar y0 mo 7-9
y0 mo 7-9	 observed SST: r=0.27584 	SST no trend: r=0.29418
		 observed SST: p=0.084913 	SST no trend: p=0.065388
y0 mo 4-6	 observed SST: r=0.21767 	SST no trend: r=0.23757
		 observed SST: p=0.17726 	SST no trend: p=0.13992
	DT INSIGNIFICANT y0 mo 4-6	p=0.17726 	 p=0.13992
PREDICTING solar y0 mo 10-12
y0 mo 10-12	 observed SST: r=-0.06548 	SST no trend: r=-0.074581
		 observed SST: p

...

y0 mo 10-12	 observed SST: r=0.29102 	SST no trend: r=0.24853
		 observed SST: p=0.068477 	SST no trend: p=0.12202
	DT INSIGNIFICANT y0 mo 10-12	p=0.06848 	 p=0.12202
y0 mo 7-9	 observed SST: r=0.2013 	SST no trend: r=0.1437
		 observed SST: p=0.21293 	SST no trend: p=0.37636
	DT INSIGNIFICANT y0 mo 7-9	p=0.21293 	 p=0.37636
PREDICTING wetdays y0 mo 1-3
y0 mo 1-3	 observed SST: r=-0.056107 	SST no trend: r=-0.042176
		 observed SST: p=0.73094 	SST no trend: p=0.7961
	DT INSIGNIFICANT y0 mo 1-3	p=0.73094 	 p=0.79610
y-1 mo 10-12	 observed SST: r=0.0060808 	SST no trend: r=0.018748
		 observed SST: p=0.97029 	SST no trend: p=0.90859
	DT INSIGNIFICANT y-1 mo 10-12	p=0.97029 	 p=0.90859
PREDICTING wetdays y0 mo 4-6
y0 mo 4-6	 observed SST: r=-0.17062 	SST no trend: r=-0.12045
		 observed SST: p=0.2925 	SST no trend: p=0.45909
	DT INSIGNIFICANT y0 mo 4-6	p=0.29250 	 p=0.45909
y0 mo 1-3	 observed SST: r=-0.072175 	SST no trend: r=-0.0048913
		 observed SST: p=0.65807 	SST no trend: p=0.9761


y0 mo 10-12	 observed SST: r=0.035901 	SST no trend: r=-0.042193
		 observed SST: p=0.82593 	SST no trend: p=0.79602
	SIGN CHANGE y0 mo 10-12	r=0.03590 	 r=-0.04219
	DT INSIGNIFICANT y0 mo 10-12	p=0.82593 	 p=0.79602
y0 mo 7-9	 observed SST: r=0.0083599 	SST no trend: r=-0.088148
		 observed SST: p=0.95917 	SST no trend: p=0.5886
	SIGN CHANGE y0 mo 7-9	r=0.00836 	 r=-0.08815
	DT INSIGNIFICANT y0 mo 7-9	p=0.95917 	 p=0.58860
PREDICTING tmin y0 mo 1-3
y0 mo 1-3	 observed SST: r=-0.43627 	SST no trend: r=-0.47052
		 observed SST: p=0.0048889 	SST no trend: p=0.0021844
y-1 mo 10-12	 observed SST: r=-0.49053 	SST no trend: r=-0.51314
		 observed SST: p=0.0013116 	SST no trend: p=0.00070931
PREDICTING tmin y0 mo 4-6
y0 mo 4-6	 observed SST: r=-0.20565 	SST no trend: r=-0.22093
		 observed SST: p=0.20301 	SST no trend: p=0.1707
	DT INSIGNIFICANT y0 mo 4-6	p=0.20301 	 p=0.17070
y0 mo 1-3	 observed SST: r=-0.27038 	SST no trend: r=-0.29389
		 observed SST: p=0.091528 	SST no trend: p=0.065667
P

y0 mo 1-3	 observed SST: r=0.17002 	SST no trend: r=0.16661
		 observed SST: p=0.29423 	SST no trend: p=0.30416
	DT INSIGNIFICANT y0 mo 1-3	p=0.29423 	 p=0.30416
y-1 mo 10-12	 observed SST: r=0.21103 	SST no trend: r=0.208
		 observed SST: p=0.19118 	SST no trend: p=0.19777
	DT INSIGNIFICANT y-1 mo 10-12	p=0.19118 	 p=0.19777
PREDICTING wetdays y0 mo 4-6
y0 mo 4-6	 observed SST: r=-0.070785 	SST no trend: r=-0.049302
		 observed SST: p=0.66426 	SST no trend: p=0.76257
	DT INSIGNIFICANT y0 mo 4-6	p=0.66426 	 p=0.76257
y0 mo 1-3	 observed SST: r=-0.0012986 	SST no trend: r=0.027813
		 observed SST: p=0.99365 	SST no trend: p=0.86473
	SIGN CHANGE y0 mo 1-3	r=-0.00130 	 r=0.02781
	DT INSIGNIFICANT y0 mo 1-3	p=0.99365 	 p=0.86473
PREDICTING wetdays y0 mo 7-9
y0 mo 7-9	 observed SST: r=-0.17118 	SST no trend: r=-0.080609
		 observed SST: p=0.29091 	SST no trend: p=0.62098
	DT INSIGNIFICANT y0 mo 7-9	p=0.29091 	 p=0.62098
y0 mo 4-6	 observed SST: r=-0.35911 	SST no trend: r=-0.2839
		 observe

y0 mo 10-12	 observed SST: r=0.37698 	SST no trend: r=0.32609
		 observed SST: p=0.016491 	SST no trend: p=0.040025
y0 mo 7-9	 observed SST: r=0.32159 	SST no trend: r=0.24198
		 observed SST: p=0.04302 	SST no trend: p=0.13249
	DT INSIGNIFICANT y0 mo 7-9	p=0.04302 	 p=0.13249
PREDICTING vpd y0 mo 1-3
y0 mo 1-3	 observed SST: r=0.32656 	SST no trend: r=0.22186
		 observed SST: p=0.039721 	SST no trend: p=0.16885
	DT INSIGNIFICANT y0 mo 1-3	p=0.03972 	 p=0.16885
y-1 mo 10-12	 observed SST: r=0.22488 	SST no trend: r=0.13856
		 observed SST: p=0.16298 	SST no trend: p=0.39383
	DT INSIGNIFICANT y-1 mo 10-12	p=0.16298 	 p=0.39383
PREDICTING vpd y0 mo 4-6
y0 mo 4-6	 observed SST: r=0.42139 	SST no trend: r=0.35486
		 observed SST: p=0.0067689 	SST no trend: p=0.024649
y0 mo 1-3	 observed SST: r=0.43114 	SST no trend: r=0.3476
		 observed SST: p=0.005479 	SST no trend: p=0.027966
PREDICTING vpd y0 mo 7-9
y0 mo 7-9	 observed SST: r=0.32787 	SST no trend: r=0.2387
		 observed SST: p=0.038892 	

y0 mo 4-6	 observed SST: r=0.26813 	SST no trend: r=0.20946
		 observed SST: p=0.094366 	SST no trend: p=0.19456
	DT INSIGNIFICANT y0 mo 4-6	p=0.09437 	 p=0.19456
y0 mo 1-3	 observed SST: r=0.27511 	SST no trend: r=0.20532
		 observed SST: p=0.085773 	SST no trend: p=0.20374
	DT INSIGNIFICANT y0 mo 1-3	p=0.08577 	 p=0.20374
PREDICTING wind y0 mo 7-9
y0 mo 7-9	 observed SST: r=-0.17984 	SST no trend: r=-0.13236
		 observed SST: p=0.26681 	SST no trend: p=0.41555
	DT INSIGNIFICANT y0 mo 7-9	p=0.26681 	 p=0.41555
y0 mo 4-6	 observed SST: r=-0.27052 	SST no trend: r=-0.22365
		 observed SST: p=0.091352 	SST no trend: p=0.16535
	DT INSIGNIFICANT y0 mo 4-6	p=0.09135 	 p=0.16535
PREDICTING wind y0 mo 10-12
y0 mo 10-12	 observed SST: r=0.13341 	SST no trend: r=0.10471
		 observed SST: p=0.41182 	SST no trend: p=0.5202
	DT INSIGNIFICANT y0 mo 10-12	p=0.41182 	 p=0.52020
y0 mo 7-9	 observed SST: r=0.019056 	SST no trend: r=-0.019213
		 observed SST: p=0.90709 	SST no trend: p=0.90633
	SIGN CHANG

# Nonforest

In [9]:
# iterate through each ecoregion, make a list of SST gradients to EXCLUDE

modeloutputfile = model_string + 'coeffExclude_sst_non_ecoprovinces.txt'
f = open(modeloutputfile, 'w') # print to this file
labels = dfframesnon['allwestUS'].columns.values # each climate variable in the columns

for iregion, ecoregname in enumerate(dfnames):
    print('++++++++ NONFOREST model for ' + dfnames[iregion]+ '\t r-value obs \t r-value DT ++++++')
    f.write('+++'+dfnames[iregion]+'\n')
    for i, label in enumerate(labels): # iterate through climate variables
        print('PREDICTING '+label)
        f.write('PREDICTING '+label)
        f.write('\n')
        # variables to compare
        # each climate variable
        tmpclim = np.asarray(dfframesnon[dfnames[iregion]][label])
        tmpclimDT = annualDetrend(tmpclim)
        # associated concurrent-season climate index (ENSO or other)
        strseason = labels[i].split(' ',1)[1] # get season label
        matchseason = climindseasons83Obs.columns.isin([strseason]) # True/False array
        tmpclimindObs = np.asarray(climindseasons83Obs.loc[:,matchseason]).flatten()
        tmpclimindDT = np.asarray(climindseasons83DT.loc[:,matchseason]).flatten()
        
        # correlation of clim vs. SST gradient
        tmppearsonObs = pearsonr(tmpclimindObs, tmpclim)
        tmprObs = tmppearsonObs.statistic
        tmppObs = tmppearsonObs.pvalue
        tmppearsonDT = pearsonr(tmpclimindDT, tmpclimDT)
        tmprDT = tmppearsonDT.statistic
        tmppDT = tmppearsonDT.pvalue
        # check if they are the same sign
        tmpsign = ((tmprObs > 0) == (tmprDT > 0))
        tmppvalue = (tmppDT < 0.1) # check significance
        print(climindseasons83Obs.columns[matchseason][0] + 
            '\t observed SST: r={:.5} \tSST no trend: r={:.5}'.format(tmprObs, tmprDT))
        print('\t\t observed SST: p={:.5} \tSST no trend: p={:.5}'.format(tmppObs, tmppDT))
        
        if tmpsign==False:
            print('\tSIGN CHANGE ' + climindseasons83Obs.columns[matchseason][0] + '\t'+
                  'r={:.5f} \t r={:.5f}'.format(tmprObs, tmprDT))
        if tmppvalue==False:
            print('\tDT INSIGNIFICANT ' + climindseasons83Obs.columns[matchseason][0] + '\t'+
                  'p={:.5f} \t p={:.5f}'.format(tmppObs, tmppDT))
        if (tmpsign==False) or (tmppvalue==False):
            f.write(str(climindseasons83Obs.columns[matchseason][0]) + '\n')
        
        # grab previous season            
        iprevseason = np.where(matchseason)
        iprevseason = int(iprevseason[0])-1
        tmpclimindObs = np.asarray(climindseasons83Obs.iloc[:,iprevseason]).flatten()
        tmpclimindDT = np.asarray(climindseasons83DT.iloc[:,iprevseason]).flatten()
        # correlation of clim vs. SST gradient
        tmppearsonObs = pearsonr(tmpclimindObs, tmpclim)
        tmprObs = tmppearsonObs.statistic
        tmppObs = tmppearsonObs.pvalue
        tmppearsonDT = pearsonr(tmpclimindDT, tmpclimDT)
        tmprDT = tmppearsonDT.statistic
        tmppDT = tmppearsonDT.pvalue
        
        # check if previous season changed correlation sign
        tmpsign = ((tmprObs > 0) == (tmprDT > 0))
        tmppvalue = (tmppDT < 0.1) # check significance
        print(climindseasons83Obs.columns[iprevseason] + 
              '\t observed SST: r={:.5} \tSST no trend: r={:.5}'.format(tmprObs, tmprDT))
        print('\t\t observed SST: p={:.5} \tSST no trend: p={:.5}'.format(tmppObs, tmppDT))
        if tmpsign==False:
            print('\tSIGN CHANGE ' + climindseasons83Obs.columns[iprevseason] + '\t'+
                  'r={:.5f} \t r={:.5f}'.format(tmprObs, tmprDT))
        if tmppvalue==False:
            print('\tDT INSIGNIFICANT ' + climindseasons83Obs.columns[iprevseason] + '\t'+
                  'p={:.5f} \t p={:.5f}'.format(tmppObs, tmppDT))
        if (tmpsign==False) or (tmppvalue==False):
            f.write(str(climindseasons83Obs.columns[iprevseason])+'\n')
f.close()

++++++++ NONFOREST model for allwestUS	 r-value obs 	 r-value DT ++++++
PREDICTING rh y0 mo 1-3
y0 mo 1-3	 observed SST: r=-0.57808 	SST no trend: r=-0.53042
		 observed SST: p=9.3651e-05 	SST no trend: p=0.00043057
y-1 mo 10-12	 observed SST: r=-0.4619 	SST no trend: r=-0.41908
		 observed SST: p=0.0026964 	SST no trend: p=0.0071103
PREDICTING rh y0 mo 4-6
y0 mo 4-6	 observed SST: r=-0.48425 	SST no trend: r=-0.43633
		 observed SST: p=0.0015445 	SST no trend: p=0.0048824
y0 mo 1-3	 observed SST: r=-0.5768 	SST no trend: r=-0.52834
		 observed SST: p=9.786e-05 	SST no trend: p=0.00045793
PREDICTING rh y0 mo 7-9
y0 mo 7-9	 observed SST: r=-0.21109 	SST no trend: r=-0.11787
		 observed SST: p=0.19105 	SST no trend: p=0.46885
	DT INSIGNIFICANT y0 mo 7-9	p=0.19105 	 p=0.46885
y0 mo 4-6	 observed SST: r=-0.34325 	SST no trend: r=-0.25958
		 observed SST: p=0.030128 	SST no trend: p=0.10576
	DT INSIGNIFICANT y0 mo 4-6	p=0.03013 	 p=0.10576
PREDICTING rh y0 mo 10-12
y0 mo 10-12	 observed SST

y0 mo 10-12	 observed SST: r=0.17484 	SST no trend: r=0.093472
		 observed SST: p=0.28056 	SST no trend: p=0.56618
	DT INSIGNIFICANT y0 mo 10-12	p=0.28056 	 p=0.56618
y0 mo 7-9	 observed SST: r=0.23688 	SST no trend: r=0.14665
		 observed SST: p=0.14111 	SST no trend: p=0.36652
	DT INSIGNIFICANT y0 mo 7-9	p=0.14111 	 p=0.36652
PREDICTING wetdays y0 mo 1-3
y0 mo 1-3	 observed SST: r=-0.45048 	SST no trend: r=-0.41764
		 observed SST: p=0.0035346 	SST no trend: p=0.0073319
y-1 mo 10-12	 observed SST: r=-0.34794 	SST no trend: r=-0.31642
		 observed SST: p=0.027803 	SST no trend: p=0.046681
PREDICTING wetdays y0 mo 4-6
y0 mo 4-6	 observed SST: r=-0.082948 	SST no trend: r=-0.076384
		 observed SST: p=0.61085 	SST no trend: p=0.63946
	DT INSIGNIFICANT y0 mo 4-6	p=0.61085 	 p=0.63946
y0 mo 1-3	 observed SST: r=-0.43628 	SST no trend: r=-0.44733
		 observed SST: p=0.0048881 	SST no trend: p=0.0038032
PREDICTING wetdays y0 mo 7-9
y0 mo 7-9	 observed SST: r=-0.099252 	SST no trend: r=-0.019846

y0 mo 4-6	 observed SST: r=-0.042062 	SST no trend: r=-0.2385
		 observed SST: p=0.79664 	SST no trend: p=0.13832
	DT INSIGNIFICANT y0 mo 4-6	p=0.79664 	 p=0.13832
PREDICTING tmean y0 mo 10-12
y0 mo 10-12	 observed SST: r=-0.060551 	SST no trend: r=-0.13046
		 observed SST: p=0.71052 	SST no trend: p=0.42233
	DT INSIGNIFICANT y0 mo 10-12	p=0.71052 	 p=0.42233
y0 mo 7-9	 observed SST: r=-0.099021 	SST no trend: r=-0.18538
		 observed SST: p=0.54325 	SST no trend: p=0.25212
	DT INSIGNIFICANT y0 mo 7-9	p=0.54325 	 p=0.25212
PREDICTING vpd y0 mo 1-3
y0 mo 1-3	 observed SST: r=0.16772 	SST no trend: r=0.14459
		 observed SST: p=0.30092 	SST no trend: p=0.37339
	DT INSIGNIFICANT y0 mo 1-3	p=0.30092 	 p=0.37339
y-1 mo 10-12	 observed SST: r=0.084829 	SST no trend: r=0.063511
		 observed SST: p=0.60276 	SST no trend: p=0.69703
	DT INSIGNIFICANT y-1 mo 10-12	p=0.60276 	 p=0.69703
PREDICTING vpd y0 mo 4-6
y0 mo 4-6	 observed SST: r=0.089541 	SST no trend: r=0.068419
		 observed SST: p=0.58269 	S

/var/folders/bh/cplk6_2163d7cqx4vgqpkzz40000gn/T/ipykernel_58417/2961295184.py:49: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  iprevseason = int(iprevseason[0])-1


y0 mo 1-3	 observed SST: r=-0.3435 	SST no trend: r=-0.33279
		 observed SST: p=0.030001 	SST no trend: p=0.035885
y-1 mo 10-12	 observed SST: r=-0.24813 	SST no trend: r=-0.23408
		 observed SST: p=0.12264 	SST no trend: p=0.14601
	DT INSIGNIFICANT y-1 mo 10-12	p=0.12264 	 p=0.14601
PREDICTING wetdays y0 mo 4-6
y0 mo 4-6	 observed SST: r=0.029739 	SST no trend: r=0.037044
		 observed SST: p=0.85546 	SST no trend: p=0.82048
	DT INSIGNIFICANT y0 mo 4-6	p=0.85546 	 p=0.82048
y0 mo 1-3	 observed SST: r=-0.21874 	SST no trend: r=-0.22241
		 observed SST: p=0.17509 	SST no trend: p=0.16778
	DT INSIGNIFICANT y0 mo 1-3	p=0.17509 	 p=0.16778
PREDICTING wetdays y0 mo 7-9
y0 mo 7-9	 observed SST: r=-0.13065 	SST no trend: r=-0.087416
		 observed SST: p=0.42166 	SST no trend: p=0.59171
	DT INSIGNIFICANT y0 mo 7-9	p=0.42166 	 p=0.59171
y0 mo 4-6	 observed SST: r=-0.012357 	SST no trend: r=0.042268
		 observed SST: p=0.93968 	SST no trend: p=0.79566
	SIGN CHANGE y0 mo 4-6	r=-0.01236 	 r=0.04227
	DT

y0 mo 1-3	 observed SST: r=-0.070181 	SST no trend: r=-0.044171
		 observed SST: p=0.66696 	SST no trend: p=0.78667
	DT INSIGNIFICANT y0 mo 1-3	p=0.66696 	 p=0.78667
PREDICTING rh y0 mo 7-9
y0 mo 7-9	 observed SST: r=-0.19732 	SST no trend: r=-0.10667
		 observed SST: p=0.2223 	SST no trend: p=0.51239
	DT INSIGNIFICANT y0 mo 7-9	p=0.22230 	 p=0.51239
y0 mo 4-6	 observed SST: r=-0.43163 	SST no trend: r=-0.36514
		 observed SST: p=0.0054199 	SST no trend: p=0.020518
PREDICTING rh y0 mo 10-12
y0 mo 10-12	 observed SST: r=0.26025 	SST no trend: r=0.25911
		 observed SST: p=0.10483 	SST no trend: p=0.10641
	DT INSIGNIFICANT y0 mo 10-12	p=0.10483 	 p=0.10641
y0 mo 7-9	 observed SST: r=0.32148 	SST no trend: r=0.32345
		 observed SST: p=0.043097 	SST no trend: p=0.041761
PREDICTING solar y0 mo 1-3
y0 mo 1-3	 observed SST: r=-0.0091544 	SST no trend: r=0.026465
		 observed SST: p=0.95529 	SST no trend: p=0.87123
	SIGN CHANGE y0 mo 1-3	r=-0.00915 	 r=0.02647
	DT INSIGNIFICANT y0 mo 1-3	p=0.955

y0 mo 10-12	 observed SST: r=-0.25177 	SST no trend: r=-0.31492
		 observed SST: p=0.11706 	SST no trend: p=0.047784
y0 mo 7-9	 observed SST: r=-0.21665 	SST no trend: r=-0.29018
		 observed SST: p=0.17934 	SST no trend: p=0.06931
PREDICTING tmin y0 mo 1-3
y0 mo 1-3	 observed SST: r=-0.52754 	SST no trend: r=-0.54376
		 observed SST: p=0.00046878 	SST no trend: p=0.00028743
y-1 mo 10-12	 observed SST: r=-0.56215 	SST no trend: r=-0.56994
		 observed SST: p=0.00016006 	SST no trend: p=0.00012358
PREDICTING tmin y0 mo 4-6
y0 mo 4-6	 observed SST: r=-0.42016 	SST no trend: r=-0.49932
		 observed SST: p=0.0069499 	SST no trend: p=0.001038
y0 mo 1-3	 observed SST: r=-0.39071 	SST no trend: r=-0.48581
		 observed SST: p=0.012678 	SST no trend: p=0.0014835
PREDICTING tmin y0 mo 7-9
y0 mo 7-9	 observed SST: r=0.01831 	SST no trend: r=-0.19775
		 observed SST: p=0.91071 	SST no trend: p=0.22127
	SIGN CHANGE y0 mo 7-9	r=0.01831 	 r=-0.19775
	DT INSIGNIFICANT y0 mo 7-9	p=0.91071 	 p=0.22127
y0 mo

y0 mo 7-9	 observed SST: r=0.37968 	SST no trend: r=0.30663
		 observed SST: p=0.015675 	SST no trend: p=0.054297
y0 mo 4-6	 observed SST: r=0.44989 	SST no trend: r=0.37999
		 observed SST: p=0.0035842 	SST no trend: p=0.015581
PREDICTING vpd y0 mo 10-12
y0 mo 10-12	 observed SST: r=0.35392 	SST no trend: r=0.31187
		 observed SST: p=0.02506 	SST no trend: p=0.050106
y0 mo 7-9	 observed SST: r=0.28641 	SST no trend: r=0.22932
		 observed SST: p=0.07318 	SST no trend: p=0.15463
	DT INSIGNIFICANT y0 mo 7-9	p=0.07318 	 p=0.15463
PREDICTING wetdays y0 mo 1-3
y0 mo 1-3	 observed SST: r=-0.040472 	SST no trend: r=-0.0013064
		 observed SST: p=0.80417 	SST no trend: p=0.99362
	DT INSIGNIFICANT y0 mo 1-3	p=0.80417 	 p=0.99362
y-1 mo 10-12	 observed SST: r=0.027614 	SST no trend: r=0.059397
		 observed SST: p=0.86569 	SST no trend: p=0.71581
	DT INSIGNIFICANT y-1 mo 10-12	p=0.86569 	 p=0.71581
PREDICTING wetdays y0 mo 4-6
y0 mo 4-6	 observed SST: r=-0.37847 	SST no trend: r=-0.34012
		 observe

In [10]:
# print last time this checker was updated
from datetime import datetime
now = datetime.now()
# dd/mm/YY H:M:S
dt_string = now.strftime("%d/%m/%Y %H:%M:%S")
print("Last checked correlations =", dt_string)
print("SST gradient =", climindnameObs)

Last checked correlations = 26/06/2026 14:54:20
SST gradient = patch125-155_nino3-34
